In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Reload data
train = pd.read_parquet("./data/LDAP-training.parquet")
test  = pd.read_parquet("./data/LDAP-testing.parquet")
df    = pd.concat([train, test], ignore_index=True)

# Binary label: 0=Benign, 1=Attack
df["BinaryLabel"] = (df["Label"] != "Benign").astype(int)

# Multi-class label
le = LabelEncoder()
df["MultiLabel"] = le.fit_transform(df["Label"])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("\nBinary distribution:\n", df["BinaryLabel"].value_counts())

Label mapping: {'Benign': np.int64(0), 'DrDoS_LDAP': np.int64(1), 'LDAP': np.int64(2), 'NetBIOS': np.int64(3)}

Binary distribution:
 BinaryLabel
0    5976
1    3570
Name: count, dtype: int64


In [3]:
from sklearn.feature_selection import VarianceThreshold

X = df.drop(columns=["Label", "BinaryLabel", "MultiLabel"])
X = X.select_dtypes(include=np.number)

selector = VarianceThreshold(threshold=0.0)
X_filtered = pd.DataFrame(selector.fit_transform(X), columns=X.columns[selector.get_support()])

print(f"Features before: {X.shape[1]}")
print(f"Features after removing zero-variance: {X_filtered.shape[1]}")
dropped = set(X.columns) - set(X_filtered.columns)
print(f"Dropped: {dropped}")

Features before: 77
Features after removing zero-variance: 65
Dropped: {'Fwd URG Flags', 'Bwd PSH Flags', 'Fwd Avg Bytes/Bulk', 'PSH Flag Count', 'Bwd URG Flags', 'FIN Flag Count', 'Bwd Avg Bulk Rate', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'ECE Flag Count', 'Bwd Avg Packets/Bulk'}


In [5]:
from sklearn.preprocessing import StandardScaler
import joblib, os
os.makedirs("models", exist_ok=True)

y_binary = df["BinaryLabel"].values
y_multi  = df["MultiLabel"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)
X_scaled = pd.DataFrame(X_scaled, columns=X_filtered.columns)

joblib.dump(scaler, "models/scaler.pkl")
X_scaled.to_parquet("data/X_scaled.parquet")
pd.Series(y_binary).to_frame("BinaryLabel").to_parquet("data/y_binary.parquet")
pd.Series(y_multi).to_frame("MultiLabel").to_parquet("data/y_multi.parquet")

print("X_scaled shape:", X_scaled.shape)
print("Scaler saved: models/scaler.pkl")
print("Data saved: data/X_scaled.parquet")

X_scaled shape: (9546, 65)
Scaler saved: models/scaler.pkl
Data saved: data/X_scaled.parquet
